# Camera, notebook controls and supervised roaming
Use a **Pi Python kernel**, locally or through Remote SSH/tunneled Jupyter. Default execution opens no hardware and installs/downloads nothing. Stop legacy servers first. Follow `docs/CAMERA_USAGE.md` for dependencies, verified model assets, calibration and physical trials. Software results cannot establish physical safety.

Camera preview, object detection and motion have separate opt-ins. Keep a physical power cutoff within reach. Use a stand for initial motor trials, then only a supervised flat enclosed floor without stairs, people or pets. Floor-color analysis is not depth sensing or guaranteed collision avoidance.

In [ ]:
from pathlib import Path
import sys
repo_root = next((p for p in (Path.cwd(), Path.cwd().parent)
                  if (p / 'rasptank' / '__init__.py').is_file()), None)
if repo_root is None:
    raise RuntimeError('Open this notebook from the repository or notebooks directory')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from rasptank import CameraController, RaspTank, RoamPolicy
from rasptank.vision import (detect_color, hsv_target, annotate, FloorObserver,
                            MobileNetSSD, detect_objects, find_object,
                            MOBILENET_PROTOTXT_SHA256, MOBILENET_WEIGHTS_SHA256)
print('IMPORT_OK: camera and vision imports open no devices')

## 1. Camera-only preview (no motors)
Enable only after checking the Pi camera and stopping other owners. This cell returns immediately with live video; later cells remain usable. Rerunning it closes the old view/tank/camera first. `Stop video` stops presentation, not capture or a manually controlled tank.

In [ ]:
RUN_CAMERA = False
from rasptank.hardware._lifecycle import release
release([globals()[name].close for name in ('tank', 'view', 'camera')
         if globals().get(name) is not None])
tank = view = camera = None
if RUN_CAMERA:
    from rasptank.notebook import show_camera
    camera = CameraController(size=(640, 480), fps=15, max_age=0.5)
    try:
        camera.start()
        view = show_camera(camera, fps=10)
    except BaseException:
        camera.close()
        raise
else:
    print('CAMERA_DISABLED')

## 2. Color target and object identification (never drive)
Color targets are not semantic identity. Semantic identification uses the explicit local VOC MobileNet-SSD model described in the usage guide; no weights are fetched by these cells. Select a supported label such as `bottle`, `chair`, `cat`, `dog` or `person`. Recognizing your particular item needs a custom adapter/reference model.

In [ ]:
RUN_COLOR_TEST = False
if RUN_COLOR_TEST:
    if camera is None:
        raise RuntimeError('Enable camera preview first')
    from IPython.display import Image, display
    frame = camera.capture()
    matches = detect_color(frame, hsv_ranges=hsv_target(0), min_area=100, label='red target')
    print(matches)
    display(Image(data=annotate(frame, matches)))
else:
    print('COLOR_TEST_DISABLED')

In [ ]:
RUN_OBJECT_TEST = False
TARGET_LABELS = ['bottle']
if RUN_OBJECT_TEST:
    if camera is None:
        raise RuntimeError('Enable camera preview first')
    from IPython.display import Image, display
    detector = MobileNetSSD(repo_root / 'models/deploy.prototxt',
                            repo_root / 'models/mobilenet_iter_73000.caffemodel',
                            prototxt_sha256=MOBILENET_PROTOTXT_SHA256,
                            weights_sha256=MOBILENET_WEIGHTS_SHA256)
    frame = camera.capture()
    matches = detect_objects(frame, detector=detector, labels=TARGET_LABELS, confidence=0.5)
    print(matches)
    display(Image(data=annotate(frame, matches)))
    # Optional bounded multi-frame search, returning structured results:
    # matches = find_object(camera, detector=detector, labels=TARGET_LABELS, timeout=5)
else:
    print('OBJECT_TEST_DISABLED')

## 3. Manual motion with live video
First secure the tank on a stand with tracks clear, verify wiring/direction and keep the cutoff ready. An opt-in command lease stops a forgotten command after 0.5 seconds; it is not a hardware emergency stop. Later separate cells can call `tank.forward(15)`, `tank.turn_left(15)`, `tank.stop()` while the view updates. Direct `tank.motors` calls bypass supervision.

In [ ]:
RUN_MANUAL_TEST = False
STAND_AND_POWER_CUTOFF_READY = False
if RUN_MANUAL_TEST:
    if camera is None or not STAND_AND_POWER_CUTOFF_READY:
        raise RuntimeError('Enable camera and complete the stand/power-cutoff checks first')
    if tank is not None:
        tank.close()
    tank = RaspTank(camera=camera, command_timeout=0.5)
    view.close()
    view = tank.show_camera(fps=10)
    tank.forward(15)  # watchdog expires this command; low duty may stall a track
else:
    print('MANUAL_MOTION_DISABLED')

## 4. Calibrated visual corridor, then finite room trial
Set `FLOOR_HSV_RANGES` from your actual illuminated floor (OpenCV H=0..179, S/V=0..255). Do not use a generic range for navigation. Verify the lower center corridor corresponds to the forward ultrasonic beam, with the camera fixed and aligned. Inspect `observer(camera.capture())` while stationary with clear floor, obstacles and poor light; any unknown must stop the tank.

Only after measuring Pi sensor age, stopping travel and watchdog latency, set the motion limits and enable a **supervised** floor trial. This example has no servo movement, rear travel or automatic model download. The recovery budget and duration are finite. `tank.stop()` cancels roaming; manual facade motion cancels it before taking over.

In [ ]:
RUN_FLOOR_CHECK = False
FLOOR_HSV_RANGES = None  # e.g. a measured list of ((Hmin,Smin,Vmin),(Hmax,Smax,Vmax))
observer = None
if RUN_FLOOR_CHECK:
    if camera is None or FLOOR_HSV_RANGES is None:
        raise RuntimeError('Enable camera and supply measured floor ranges first')
    observer = FloorObserver(FLOOR_HSV_RANGES, roi=(0.3, 0.55, 0.7, 0.95))
    print(observer(camera.capture()))
else:
    print('FLOOR_CHECK_DISABLED')

In [ ]:
RUN_ROAM_TEST = False
MOTION_LIMITS_VERIFIED = False
SUPERVISED_FLAT_ROOM_READY = False
if RUN_ROAM_TEST:
    if camera is None or observer is None or not (MOTION_LIMITS_VERIFIED and SUPERVISED_FLAT_ROOM_READY):
        raise RuntimeError('Complete camera/floor calibration, measured limits and room safety checks first')
    if tank is not None:
        tank.close()
    tank = RaspTank(camera=camera, enable_ultrasonic=True, command_timeout=0.5)
    if view is not None:
        view.close()
    view = tank.show_camera(fps=10)
    policy = RoamPolicy(speed=20, stop_distance=0.5, resume_distance=0.6,
                        max_age=0.3, pulse=0.1, max_turns=12, max_duration=10)
    session = tank.start_roaming(observer=observer, policy=policy)
    print(session.status())  # returns immediately; inspect tank.roaming_status() in another cell
else:
    print('ROAMING_DISABLED')

## Stop and cleanup
Run this cell before leaving the notebook or switching owners. A latched fault remains visible in `tank.roaming_status()`; inspect its cause before explicitly calling `tank.reset_drive_fault()`. A stuck custom observer must exit before restarting autonomy. Kernel/OS failure still requires the physical cutoff.

In [ ]:
release([globals()[name].close for name in ('tank', 'view', 'camera')
         if globals().get(name) is not None])
tank = view = camera = None
print('CLEANUP_DONE')